# ⚽ Mission 12: Soccer Vision Lab — Build an AI Tactical Analyst

## 📖 Mission Story
Welcome back, Coach Artin! 🎉

During the first eleven missions, you gradually learned how to become an AI-powered soccer coach. You started with Python programming, then learned how to organize data with Pandas, visualize player statistics, build professional Streamlit dashboards, and even consult Gemini AI to answer tactical questions.

But there has always been one important limitation: every dashboard you built assumed that someone had already collected the player statistics. Professional soccer clubs don't work that way. Instead of receiving ready-made numbers, they begin with a match video!

Today, you are going to build your own sports analytics system. By the end of this mission, your Soccer AI Coach will no longer depend on manually entered statistics—it will create its own statistics directly from match footage. Welcome to the world of **Computer Vision**!

---

## 🎯 Learning Objectives
* ✅ Explain how computers represent videos.
* ✅ Read soccer videos using OpenCV (`cv2`).
* ✅ Upload videos inside Streamlit using `st.file_uploader()`.
* ✅ Detect soccer players using YOLO (`ultralytics`).
* ✅ Understand the difference between object detection and object tracking.
* ✅ Convert pixel coordinates into real soccer pitch coordinates ($120 \times 80$ yards).
* ✅ Generate professional tactical heatmaps using `mplsoccer`.
* ✅ Ask Gemini AI to analyze player movement as an elite UEFA Pro Analyst.
* ✅ Combine Computer Vision, Data Science, Visualization, and AI into one complete application.

---

### 🟢 Phase 1: Understanding Soccer Heatmaps

Before we teach the computer how to collect movement from video, we first learn how to visualize movement. Every time a player touches the ball or moves into space, we record their $(X, Y)$ coordinates and map them on a pitch.

Let's execute the full pitch generation and position simulation below:

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import numpy as np

# 1. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# 2. Simulate ball touch data (X: 0 to 120 yards, Y: 0 to 80 yards)
player = "Artin"
position = "Right Defender"

if position == "Left Winger":
    x_coordinates = [10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105]
    y_coordinates = [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]

elif position == "Striker":
    x_coordinates = [70,75,80,85,90,95,100,102,105,108,110,112,115,108,104,100,95,90,88,110]
    y_coordinates = [35,38,40,42,40,38,35,37,40,42,39,36,40,45,48,50,45,42,38,35]

elif position == "Midfielder":
    x_coordinates = [35,40,45,50,55,60,65,70,60,55,50,45,40,55,65,75,70,60,50,45]
    y_coordinates = [25,30,35,40,45,40,35,30,25,20,25,30,35,45,50,45,40,35,30,25]

elif position == "Right Defender":
    # Modern attacking fullback: defends deep but frequently overlaps on the right wing
    x_coordinates = [15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110]
    y_coordinates = [70,72,68,70,72,74,76,74,72,70,68,70,72,74,76,72,68,65,60,55]

# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw individual touch points on top
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(
    f"{player}'s Soccer Heatmap - {position}",
    fontsize=18,
    fontweight='bold',
    pad=15
)
plt.show()


### 📊 Phase 2: From Manual Coordinates to Real Player Data

Professional tracking systems record thousands of rows of player trajectory data. We structure movement data into DataFrames and save them into `.csv` files.

In [ ]:
%%writefile phase2_csv_loader.py
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Generate synthetic trajectory data
data = {
    'frame': list(range(1, 11)),
    'x': [18, 20, 21, 25, 30, 42, 55, 68, 72, 85],
    'y': [52, 53, 54, 50, 48, 45, 40, 38, 35, 30]
}

# Convert dictionary into DataFrame & save as CSV
df = pd.DataFrame(data)
df.to_csv("player_movement.csv", index=False)
print("✅ CSV file created!")

# Read CSV file and draw pitch heatmap
movement_df = pd.read_csv("player_movement.csv")
print(movement_df)

pitch = Pitch(pitch_type='statsbomb', pitch_color='#101010', line_color='#888888')
fig, ax = pitch.draw(figsize=(10, 7))
pitch.kdeplot(movement_df['x'], movement_df['y'], ax=ax, cmap='magma', fill=True, alpha=0.6)
plt.title("Artin Movement Heatmap")
plt.savefig("phase2_heatmap.png", bbox_inches='tight')
print("✅ Saved phase2_heatmap.png!")


### 🎥 Phase 3: Teaching the Computer to Read Soccer Videos with OpenCV

A video is simply a sequence of still images called **frames**. OpenCV (`cv2`) opens the video stream, retrieves metadata (Resolution, FPS, Total Frames), and extracts individual frames for analysis.

In [ ]:
%%writefile phase3_opencv_intro.py
import cv2

def inspect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video: {video_path}")
        return
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    print("📹 Video Information")
    print("-------------------")
    print(f"Resolution: {width} x {height}")
    print(f"FPS: {fps:.2f}")
    print(f"Total Frames: {frame_count}")
    print(f"Duration: {duration:.2f} seconds")
    
    success, frame = cap.read()
    if success:
        cv2.imwrite("first_frame.jpg", frame)
        print("✅ First frame saved as first_frame.jpg")
    cap.release()

# inspect_video("sample_soccer.mp4")


### 💻 Phase 4: Building a Soccer Video Upload Interface with Streamlit

We use Streamlit's `st.file_uploader()` and `tempfile` module to allow seamless video upload and pass the temporary file path directly to OpenCV.

In [ ]:
%%writefile phase4_video_upload.py
import streamlit as st
import cv2
import tempfile

st.title("⚽ Soccer Video Analyzer")
uploaded_file = st.file_uploader("Upload Match Video", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    st.video(uploaded_file)
    
    # Create temporary file
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    video_path = tfile.name
    
    # Open video using OpenCV
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    st.subheader("📹 Video Metadata")
    st.write(f"**Resolution:** {width} x {height}")
    st.write(f"**FPS:** {fps:.2f}")
    st.write(f"**Total Frames:** {frame_count}")
    if fps > 0:
        st.write(f"**Duration:** {frame_count / fps:.2f} seconds")
    cap.release()


### 🤖 Phase 5: YOLO — Detection & Object Tracking

YOLO (*You Only Look Once*) detects players (Class `0` = person) and tracks them across consecutive frames using persistent track IDs.

In [ ]:
%%writefile phase5_yolo_demo.py
from ultralytics import YOLO

# Load pretrained nano model
model = YOLO("yolov8n.pt")
print("✅ YOLO Model Loaded Successfully!")

# Tracking invocation example (uncomment when video file is ready):
# results = model.track("sample_soccer.mp4", persist=True)
# print("Tracking process completed.")


### 🟢 Phase 6: Coordinate Mapping — From Pixel Space to Pitch Space

To turn video coordinates ($1280 \times 720$ pixels) into tactical positions, we scale them proportionally onto the $120 \times 80$ yard pitch, clamping any out-of-bounds bounding boxes.

In [ ]:
%%writefile tracking_utils.py
def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))

def map_pixels_to_pitch(pixel_x, pixel_y, frame_width=1280, frame_height=720, pitch_length=120.0, pitch_width=80.0):
    """
    Maps camera pixel coordinates to standard StatsBomb pitch dimensions.
    """
    raw_pitch_x = (pixel_x / frame_width) * pitch_length
    raw_pitch_y = (pixel_y / frame_height) * pitch_width
    
    pitch_x = clamp(raw_pitch_x, 0.0, pitch_length)
    pitch_y = clamp(raw_pitch_y, 0.0, pitch_width)
    
    return round(pitch_x, 2), round(pitch_y, 2)

# Quick unit testing
px, py = map_pixels_to_pitch(640, 360)
print(f"Pixels (640, 360) -> Pitch ({px} yds, {py} yds)")


### 🏆 Phase 7 & 8: Final Boss Project — Complete Application (`app.py`)

This brings all 8 phases together into a Streamlit application that handles video ingest, player tracking, heatmap rendering, and Gemini AI tactical analysis.

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import pandas as pd
import numpy as np
import tempfile
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import google.generativeai as genai
from ultralytics import YOLO
from tracking_utils import map_pixels_to_pitch

st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")

st.title("⚽ Soccer Vision Lab: AI Tactical Analyst")
st.write("Upload match footage to track player movement, draw professional heatmaps, and generate UEFA Pro tactical insights.")

st.sidebar.header("⚙️ Tracking Settings")
target_player_id = st.sidebar.number_input("Target Track ID", min_value=1, value=1, step=1)

# Initialize Gemini Model
try:
    genai.configure(api_key=st.secrets["GEMINI_API_KEY"])
    ai_model = genai.GenerativeModel("gemini-1.5-flash")
except Exception:
    st.sidebar.warning("Provide GEMINI_API_KEY in st.secrets to enable AI Analysis.")
    ai_model = None

uploaded_file = st.file_uploader("Upload Video Clip (.mp4, .mov)", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    
    st.success("Video uploaded successfully. Extracting tracking data...")
    
    cap = cv2.VideoCapture(tfile.name)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    yolo_model = YOLO('yolov8n.pt')
    tracking_records = []
    frame_idx = 0
    progress_bar = st.progress(0)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= 150: # Limit frames for interactive demo
            break
            
        frame_idx += 1
        if total_frames > 0:
            progress_bar.progress(min(frame_idx / min(total_frames, 150), 1.0))
            
        results = yolo_model.track(frame, persist=True, verbose=False)[0]
        
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()
            
            for box, track_id, cls in zip(boxes, track_ids, clss):
                if int(cls) == 0 and int(track_id) == target_player_id:
                    x1, y1, x2, y2 = box
                    center_x = (x1 + x2) / 2.0
                    center_y = y2 # Bottom feet position
                    
                    pitch_x, pitch_y = map_pixels_to_pitch(
                        center_x, center_y, frame_width, frame_height
                    )
                    
                    tracking_records.append({
                        "frame": frame_idx,
                        "pitch_x": pitch_x,
                        "pitch_y": pitch_y
                    })
    
    cap.release()
    st.success("Tracking process completed!")
    
    if tracking_records:
        df_tracking = pd.DataFrame(tracking_records)
        
        col1, col2 = st.columns([1, 1])
        
        with col1:
            st.subheader("📍 Tactical Movement Heatmap")
            pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
            fig, ax = pitch.draw(figsize=(8, 5))
            bin_statistic = pitch.bin_statistic(
                df_tracking['pitch_x'], df_tracking['pitch_y'], statistic='count', bins=(12, 8)
            )
            pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', alpha=0.6)
            pitch.scatter(df_tracking['pitch_x'], df_tracking['pitch_y'], c='black', s=20, ax=ax)
            st.pyplot(fig)
            
        with col2:
            st.subheader("📋 Movement Summary")
            avg_x = df_tracking['pitch_x'].mean()
            avg_y = df_tracking['pitch_y'].mean()
            st.metric("Average Position X", f"{avg_x:.1f} yards")
            st.metric("Average Position Y", f"{avg_y:.1f} yards")
            st.metric("Total Detected Frames", len(df_tracking))
            
        st.divider()
        st.subheader("🤖 UEFA Pro AI Tactical Report")
        
        if ai_model and st.button("Generate AI Tactical Assessment"):
            prompt = f"""
            You are an elite UEFA Pro soccer analyst.
            
            Here is a player's positional tracking summary for Player #{target_player_id}:
            - Average X Coordinate (Pitch Length 0-120 yards): {avg_x:.1f}
            - Average Y Coordinate (Pitch Width 0-80 yards): {avg_y:.1f}
            - Total Logged Tracking Frames: {len(df_tracking)}
            - Min X: {df_tracking['pitch_x'].min():.1f}, Max X: {df_tracking['pitch_x'].max():.1f}
            - Min Y: {df_tracking['pitch_y'].min():.1f}, Max Y: {df_tracking['pitch_y'].max():.1f}
            
            Provide a structured assessment including:
            1. Strengths observed from positioning
            2. Weaknesses / Positioning Flaws
            3. Tactical recommendations for future matches
            4. Professional player comparison
            """
            with st.spinner("Consulting AI Tactical Analyst..."):
                response = ai_model.generate_content(prompt)
                st.markdown(response.text)
    else:
        st.warning(f"No tracking records detected for Track ID #{target_player_id}. Try adjusting the Track ID.")
